# Day 6 ? Conversion Prediction and Behavioral Modeling

This notebook defines a defensible conversion-prediction task for the Digital Behavior Observatory dataset and evaluates a small set of interpretable models. The goal is to identify which behavioral patterns are most associated with conversion while preventing target leakage and temporal leakage.

## Prediction question

Which session-level behaviors indicate that a visitor is likely to complete a transaction in this dataset?

## Target definition

- Class 1: a session with at least one observed transaction (`converted == True`)
- Class 0: a session with no recorded transaction
- Prediction unit: the session, because the dataset is organized around GA visitor and visit IDs and the conversion flag is defined at the session level.

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
from src.modeling import build_modeling_dataset, load_session_features, load_user_behavior

session_features = load_session_features('data/processed/session_features.csv')
user_behavior = load_user_behavior('data/processed/user_behavior_features.csv')
modeling_table = build_modeling_dataset(session_features, user_behavior)
print(f"rows: {len(modeling_table):,}")
print(f"conversion_rate: {modeling_table['target'].mean():.4f}")
modeling_table.head()


rows: 2,556
conversion_rate: 0.0168


,entity_id,event_count,unique_event_count,unique_page_count,unique_action_count,navigation_depth,repeated_action_count,device_type,platform,browser,...,is_weekend,time_of_day,session_count,avg_session_duration_seconds,events_per_session,avg_navigation_depth,avg_unique_event_count,return_frequency,repeated_action_frequency,target
0,0004915997121163857:1501620292,1,1,1,0,1,0,mobile,iOS,Safari,...,False,evening,1,0.000,1.0,1.0,1.0,1.0,0.0,0
1,0013182148727187801:1501580378,7,1,4,0,7,0,tablet,iOS,Safari,...,False,morning,1,128.233,7.0,7.0,1.0,1.0,0.0,0
2,0015547161740087073:1501625964,1,1,1,0,1,0,tablet,iOS,Safari,...,False,late night,1,0.000,1.0,1.0,1.0,1.0,0.0,0
3,0017557475227585700:1501610006,1,1,1,0,1,0,desktop,Macintosh,Chrome,...,False,evening,1,0.000,1.0,1.0,1.0,1.0,0.0,0
4,0018642067897203467:1501622515,2,1,2,0,2,0,mobile,iOS,Safari,...,False,late night,1,14.920,2.0,2.0,1.0,1.0,0.0,0


## Feature Leakage Audit

Before modeling we explicitly check whether each candidate signal is available at the time the session is scored. Variables that directly encode the transaction outcome or that are unique identifiers are excluded.

In [2]:
from src.modeling import get_feature_leakage_audit
audit = get_feature_leakage_audit()
audit


,feature,included,reason
0,session_id,False,"Unique session identifier is an entity ID, not..."
1,user_id,False,User ID is not a causal feature and would leak...
2,source_session_id,False,"This is the source session identifier, not a b..."
3,transaction_count,False,This is the direct conversion metric and would...
4,converted,False,This is the prediction target; it must not be ...
5,session_start,False,Start timestamp is metadata and not behavioral...
6,session_end,False,The session end timestamp is not available bef...
7,session_date,False,This reflects the same sample window and can l...
8,week,False,Calendar buckets are not behavioral drivers an...
9,month,False,Month level is a sampling artifact rather than...


In [3]:
from src.modeling import main
main()
print('Model artifacts written to data/processed and models/')


Built modeling table with 2,556 sessions and 0.0168 conversion rate.
                 model  validation_threshold  validation_roc_auc  validation_average_precision  validation_f1  validation_precision  validation_recall  validation_balanced_accuracy  test_roc_auc  test_average_precision  test_f1  test_precision  test_recall  test_balanced_accuracy
   logistic_regression              0.444508            0.991799                      0.561012       0.636364              0.500000              0.875                      0.930542      0.987630                0.752044 0.518519        0.388889     0.777778                0.877954
         random_forest              0.768387            0.978628                      0.427908       0.444444              0.400000              0.500                      0.744036      0.986304                0.486356 0.454545        0.384615     0.555556                0.769825
hist_gradient_boosting              0.017089            0.987575                      0.

In [4]:
import pandas as pd
from pathlib import Path
repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
comparison = pd.read_csv(repo_root / 'data/processed/conversion_model_comparison.csv')
comparison.sort_values(['validation_average_precision', 'validation_roc_auc'], ascending=False).reset_index(drop=True)


,model,validation_threshold,validation_roc_auc,validation_average_precision,validation_f1,validation_precision,validation_recall,validation_balanced_accuracy,test_roc_auc,test_average_precision,test_f1,test_precision,test_recall,test_balanced_accuracy
0,logistic_regression,0.444508,0.991799,0.561012,0.636364,0.500000,0.875,0.930542,0.987630,0.752044,0.518519,0.388889,0.777778,0.877954
1,random_forest,0.768387,0.978628,0.427908,0.444444,0.400000,0.500,0.744036,0.986304,0.486356,0.454545,0.384615,0.555556,0.769825
2,hist_gradient_boosting,0.017089,0.987575,0.399218,0.615385,0.444444,1.000,0.990060,0.964657,0.274910,0.258065,0.181818,0.444444,0.704330


## Interpretable model explanation

The strongest configuration is a logistic regression model, which offers a linear and interpretable view of the conversion signal. The coefficients show which behaviors increase or decrease conversion likelihood relative to the baseline.

In [5]:
import pickle
import pandas as pd
from pathlib import Path
from src.modeling import build_error_analysis

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
with open(repo_root / 'models/best_conversion_model.pkl', 'rb') as handle:
    model = pickle.load(handle)

feature_columns = [col for col in pd.read_csv(repo_root / 'data/processed/conversion_modeling_table.csv').columns if col not in {'entity_id', 'target'}]
preprocessor = model.named_steps['preprocessor']
transformed_feature_names = preprocessor.get_feature_names_out()
coefs = model.named_steps['model'].coef_.ravel()
importance = pd.DataFrame({'feature': transformed_feature_names, 'coefficient': coefs})
importance['abs_coefficient'] = importance['coefficient'].abs()
importance.sort_values('abs_coefficient', ascending=False).head(10)


,feature,coefficient,abs_coefficient
287,categorical__exit_page_/ordercompleted.html,5.377737,5.377737
225,categorical__exit_page_/google+redesign/drinkware,2.537471,2.537471
129,categorical__entry_page_/google+redesign/shop+...,2.459482,2.459482
286,categorical__exit_page_/myaccount.html?mode=vi...,2.162449,2.162449
143,categorical__exit_page_/basket.html,-1.626488,1.626488
285,categorical__exit_page_/myaccount.html?mode=vi...,1.560388,1.560388
283,categorical__exit_page_/myaccount.html?mode=bi...,1.411266,1.411266
23,categorical__platform_Chrome OS,1.109941,1.109941
288,categorical__exit_page_/payment.html,-0.975548,0.975548
18,categorical__device_type_desktop,0.861359,0.861359


In [6]:
import pandas as pd
import pickle
from pathlib import Path
from src.modeling import build_error_analysis
repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
modeling_table = pd.read_csv(repo_root / 'data/processed/conversion_modeling_table.csv')
with open(repo_root / 'models/best_conversion_model.pkl', 'rb') as handle:
    model = pickle.load(handle)
error_table = build_error_analysis(modeling_table, model, 0.5)
error_table.head(10)


,entity_id,target,predicted_probability,predicted_label,is_error,error_type
1498,5835011534221003810:1501620679,1,0.444508,0,True,false_negative
34,0126612099954375841:1501606687,1,0.408094,0,True,false_negative
2200,8730493892218660083:1501625561,1,0.379293,0,True,false_negative
73,024507252193437459:1501595478,1,0.105911,0,True,false_negative
536,1933634293342529288:1501605330,0,0.999996,1,True,false_positive
436,1589021726696497303:1501615026,0,0.998539,1,True,false_positive
1744,6884489731245835935:1501600079,0,0.992528,1,True,false_positive
986,3754779580225027430:1501646820,0,0.985291,1,True,false_positive
342,1263812856049069716:1501612211,0,0.984951,1,True,false_positive
1731,6819686597729874393:1501647240,0,0.973784,1,True,false_positive


## Business interpretation

- Behavior volume and product exploration matter more than simple device context alone.
- Sessions with deeper page engagement, richer action diversity, and more purposeful browsing are more likely to convert.
- High-performing sessions are usually characterized by sustained navigation and deliberate product interaction rather than passive browsing.
- The Day 4 user-behavior context and the Day 5 anomaly diagnostics help explain why a small number of unusually intense or repetitive sessions can be highly predictive while also being less stable as a general production rule.

## SQL analysis

The SQL file in `sql/conversion_prediction.sql` reproduces the key business questions in a database-friendly form: conversion rate by device type, time-of-day patterns, and user-context summary metrics.